# BOE Constants Scraper — RO Calculation
Extract all quarterly regulatory constants from a saved BOE HTML file (e.g. `BOE-A-2026-226`)
and build the `constants` dict ready for `ro_calculation.config`.

**Workflow:**
1. Load and parse the BOE HTML with BeautifulSoup
2. Extract market inputs (OMIP, CO₂, MIBGAS) and tariff tables
3. Clean Spanish numeric formats and normalise units
4. Build RL-level cost parameter tables (RL6–RL11)
5. Assemble `constants` dict for a chosen RL level
6. Run and validate the RO formula
7. Export outputs for reuse

## 1 · Set Up Notebook Environment and Inputs

In [ ]:
from __future__ import annotations

import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

# ── Paths ──────────────────────────────────────────────────────────────────────
PROJECT_DIR = Path.cwd().parent
OUTPUT_DIR = PROJECT_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# ── BOE source (direct URL) ───────────────────────────────────────────────────
# Resolucion de 2 de julio de 2026 (BOE-A-2026-14552, BOE num. 162, Sec. III, pag. 92816):
# "actualizan los valores de la retribucion a la operacion... tercer trimestre natural del ano 2026".
# A local copy also lives at docs/boe/BOE-A-2026-14552 Resolucion 2026-07-02 valores RO.pdf
# (excluded from Claude's read access by .claudeignore / settings.json by default).
BOE_URL = "https://www.boe.es/diario_boe/txt.php?id=BOE-A-2026-14552"

# ── Configuration ──────────────────────────────────────────────────────────────
# RL level used by the plant (RL6 … RL11). IT-01144 uses RL11 — confirmed by matching
# every RL-dependent toll constant in ro_calculation/config.py's STATIC_CONSTANTS against
# the RL11 row of this resolution's tables.
TARGET_RL_LEVEL = "RL11"

# Installation-specific constants for IT-01144, 2026 (from ro_parameters.xlsx).
# IVPEE is the one exception: it is NOT a per-installation constant, it's a quarterly
# tax rate (see data/IVPEE.xlsx) — 4.9% for q3_26.
PLANT_CONSTANTS = {
    "Vc":           2.606,    # MWhPCI/MWhE — specific to installation type
    "PCI":          0.9,      # MWhPCI/MWhPCS conversion
    "C_OYM_OTROS":  20.999,   # €/MWhE — O&M and other costs
    "V_CO2":        0.406,    # tCO2/MWhE
    "RINV":         15520,    # €/MW — investment remuneration
    "V_IVPEE_RI":   5158,     # h — installation-specific hours
    "V_AC":         0.0,      # p.u.
    "P_OTROS":      0.0,      # €/MWhE
    "V_HV":         1.0289,   # MWhPCI/MWhE
    "V_HF":         0.611,    # €/MWhE
    "I_HE":         0.0,      # €/MWhE
    "IVPEE":        4.9,      # % — q3_26 rate per data/IVPEE.xlsx, not a BOE table value
}

print(f"BOE URL: {BOE_URL}")
print(f"Target RL level: {TARGET_RL_LEVEL}")

## 2 · Parse BOE HTML and Locate Target Tables

In [20]:
response = requests.get(BOE_URL, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# Collect all <table> elements found in the document body
all_tables = soup.find_all("table", class_="tabla")
print(f"Total <table class='tabla'> elements found: {len(all_tables)}")

# Print first row of each table to identify structure
for idx, tbl in enumerate(all_tables[:15]):          # first 15 only — Annexes are huge
    rows = tbl.find_all("tr")
    cells = [td.get_text(" ", strip=True) for td in rows[0].find_all(["th", "td"])] if rows else []
    row2 = [td.get_text(" ", strip=True) for td in rows[1].find_all(["th", "td"])] if len(rows) > 1 else []
    print(f"\nTable {idx:>2}: {len(rows)} rows | first-row cells: {cells}")
    if row2:
        print(f"          second-row cells: {row2}")

Total <table class='tabla'> elements found: 12

Table  0: 6 rows | first-row cells: ['Tipo de futuro', 'Precio medio (€/MWh E )']
          second-row cells: ['Futuro anual de OMIP con liquidación en el año 2026.', '61,091']

Table  1: 7 rows | first-row cells: ['Nivel del peaje de red local', 'Estimación del precio de gas natural (€/MWh PCS )']
          second-row cells: ['RL6', '36,266']

Table  2: 6 rows | first-row cells: ['Tipo de producto', 'Precio medio (€/MWh PCS )']
          second-row cells: ['Producto anual de MIBGAS con entrega en el año 2026.', '31,181']

Table  3: 3 rows | first-row cells: ['Tasas y cuotas', 'Valor']
          second-row cells: ['Tasa de la Comisión Nacional de los Mercados y la Competencia (CNMC).', '0,140\u202f%']

Table  4: 7 rows | first-row cells: ['Nivel del peaje de red local', 'Término fijo de capacidad del peaje de otros costes de regasificación (€/(kWh/día)/año) (1)']
          second-row cells: ['RL6', '-0,038838']

Table  5: 3 rows | first-r

## 3 · Clean Spanish Numeric Formats and Standardise Units

In [21]:
def sp_float(text: str) -> float:
    """Convert a Spanish-formatted number string to float.

    Handles: non-breaking spaces, thousand-separator dots, decimal commas,
    trailing '%' symbols, and surrounding whitespace.
    Examples: '0,140 %' -> 0.14,  '1.354' -> 1.354,  '-0,038838' -> -0.038838
    """
    text = text.replace("\xa0", "").strip()           # remove non-breaking space
    text = re.sub(r"\s*%\s*$", "", text)              # strip trailing %
    # Spanish thousands separator is '.', decimal is ','
    # Heuristic: if both '.' and ',' present, '.' is thousands sep
    if "," in text and "." in text:
        text = text.replace(".", "").replace(",", ".")
    else:
        text = text.replace(",", ".")
    return float(text)


def table_to_df(tbl_tag) -> pd.DataFrame:
    """Convert a BeautifulSoup <table> tag to a tidy DataFrame (all strings)."""
    rows = tbl_tag.find_all("tr")
    data = []
    for tr in rows:
        cells = [td.get_text(" ", strip=True) for td in tr.find_all(["th", "td"])]
        if cells:
            data.append(cells)
    if not data:
        return pd.DataFrame()
    max_cols = max(len(r) for r in data)
    padded = [r + [""] * (max_cols - len(r)) for r in data]
    return pd.DataFrame(padded)


# Quick smoke-test
print(sp_float("0,140 %"))     # → 0.14
print(sp_float("1,354 %"))     # → 1.354
print(sp_float("-0,038838"))   # → -0.038838
print(sp_float("0,091488"))    # → 0.091488
print(sp_float("21,8"))        # → 21.8

0.14
1.354
-0.038838
0.091488
21.8


## 4 · Extract TASA_CNMC, TASA_GTS — "Tasas y cuotas" Table

In [22]:
# The BOE "Tercero" section contains a 2-column table:
#   col 0: description, col 1: value
# We search for rows containing CNMC and GTS keywords.

def find_row_value(tbl_tag, keyword: str) -> float | None:
    """Return the float value from the first row whose first cell contains `keyword`."""
    for tr in tbl_tag.find_all("tr"):
        cells = [td.get_text(" ", strip=True) for td in tr.find_all(["th", "td"])]
        if len(cells) >= 2 and keyword.lower() in cells[0].lower():
            try:
                return sp_float(cells[1])
            except (ValueError, IndexError):
                pass
    return None


# Search all tables for CNMC / GTS
TASA_CNMC = None
TASA_GTS  = None

for tbl in all_tables:
    v = find_row_value(tbl, "CNMC")
    if v is not None:
        TASA_CNMC = v
    v = find_row_value(tbl, "Gestor T")   # "Gestor Técnico del Sistema"
    if v is not None:
        TASA_GTS = v

print(f"TASA_CNMC = {TASA_CNMC}")   # expected: 0.14 (0,140 %)
print(f"TASA_GTS  = {TASA_GTS}")    # expected: ~1.354 (1,354 %)

TASA_CNMC = 0.14
TASA_GTS  = 1.354


## 5 · Extract TC_SA, TV_SA — Peaje Salida Red Transporte

In [23]:
# TC_SA: "término fijo de capacidad del peaje de salida de la red de transporte"
# TV_SA: "término variable de volumen del peaje de salida de la red de transporte"

TC_SA = None
TV_SA = None

for tbl in all_tables:
    v = find_row_value(tbl, "término fijo de capacidad del peaje de salida")
    if v is not None:
        TC_SA = v
    v = find_row_value(tbl, "término variable de volumen del peaje de salida")
    if v is not None:
        TV_SA = v

print(f"TC_SA = {TC_SA}")   # expected: 0.091488
print(f"TV_SA = {TV_SA}")   # expected: 0.0

TC_SA = 0.091488
TV_SA = 0.0


## 6 · Build RL-Level Cost Parameter Tables (RL6–RL11)

The BOE "Tercero" section has three tables with RL-indexed rows:

| Table | Constant | Description |
|-------|----------|-------------|
| OCR (2-col) | `TC_OCR_J_BASE` | Término fijo peaje otros costes de regasificación |
| RL multi (4-col) | `TC_RL_J_BASE_2`, `TC_RL_J_BASE_1`, `TV_RL_J_BASE` | Término fijo capacidad RL (cargos unitarios, TC, TV) |
| Almacenamiento (2-col) | `D_RS`, `C_AS_BASE`, `C_AS_EXTRA`, `C_I_BASE`, `C_E_BASE` | Almacenamiento subterráneo & días de reserva |

In [24]:
RL_LEVELS = ["RL6", "RL7", "RL8", "RL9", "RL10", "RL11"]

# ── OCR table: 2 columns (RL level | TC_OCR_J_BASE) ───────────────────────────
ocr_data: dict[str, float] = {}

for tbl in all_tables:
    df = table_to_df(tbl)
    if df.empty or df.shape[1] < 2:
        continue
    for _, row in df.iterrows():
        label = str(row.iloc[0]).upper().replace(" ", "").replace("-", "")
        for rl in RL_LEVELS:
            if label.startswith(rl):
                try:
                    ocr_data[rl] = sp_float(str(row.iloc[1]))
                except ValueError:
                    pass

df_ocr = pd.DataFrame.from_dict(ocr_data, orient="index", columns=["TC_OCR_J_BASE"])
df_ocr.index.name = "RL_level"
print("OCR table:")
print(df_ocr)

# ── RL 4-column table: RL level | TC_RL_J_BASE_2 | TC_RL_J_BASE_1 | TV_RL_J_BASE
# col mapping per comments in ro_calculation/config.py:
#   col 1 → TC_RL_J_BASE_2  (término fijo de capacidad)
#   col 2 → TC_RL_J_BASE_1  (cargos unitarios)
#   col 3 → TV_RL_J_BASE    (término variable)
rl_data: dict[str, dict] = {}

for tbl in all_tables:
    df = table_to_df(tbl)
    if df.empty or df.shape[1] < 4:
        continue
    for _, row in df.iterrows():
        label = str(row.iloc[0]).upper().replace(" ", "").replace("-", "")
        for rl in RL_LEVELS:
            if label.startswith(rl):
                try:
                    rl_data[rl] = {
                        "TC_RL_J_BASE_2": sp_float(str(row.iloc[1])),
                        "TC_RL_J_BASE_1": sp_float(str(row.iloc[2])),
                        "TV_RL_J_BASE":   sp_float(str(row.iloc[3])),
                    }
                except (ValueError, IndexError):
                    pass

df_rl = pd.DataFrame.from_dict(rl_data, orient="index")
df_rl.index.name = "RL_level"
print("\nRL table (4-col):")
print(df_rl)

OCR table:
          TC_OCR_J_BASE
RL_level               
RL6            0.006152
RL7            0.006144
RL8            0.006141
RL9            0.006140
RL10           0.006140
RL11           0.006140

RL table (4-col):
          TC_RL_J_BASE_2  TC_RL_J_BASE_1  TV_RL_J_BASE
RL_level                                              
RL6             0.006152        1.344022      0.001468
RL7             0.006144        0.838860      0.001013
RL8             0.006141        0.441660      0.000699
RL9             0.006140        0.160022      0.000488
RL10            0.006140        0.153011      0.000349
RL11            0.006140        0.139655      0.000046


## 7 · Extract Almacenamiento Constants (D_RS, C_AS_BASE, C_AS_EXTRA, C_I_BASE, C_E_BASE)

In [25]:
# Keywords (Spanish) → constant key mapping
STORAGE_KEYWORDS: dict[str, str] = {
    "días de existencias":          "D_RS",
    "canon de almacenamiento":      "C_AS_BASE",
    "prima de las subasta":         "C_AS_EXTRA",   # prima subastas adjudicación de capacidad
    "canon de inyección":           "C_I_BASE",
    "canon de extracción":          "C_E_BASE",
}

storage_values: dict[str, float] = {}

for tbl in all_tables:
    for tr in tbl.find_all("tr"):
        cells = [td.get_text(" ", strip=True) for td in tr.find_all(["th", "td"])]
        if len(cells) < 2:
            continue
        desc = cells[0].lower()
        for kw, const_key in STORAGE_KEYWORDS.items():
            if kw in desc and const_key not in storage_values:
                try:
                    storage_values[const_key] = sp_float(cells[1])
                except ValueError:
                    pass

D_RS      = storage_values.get("D_RS")
C_AS_BASE = storage_values.get("C_AS_BASE")
C_AS_EXTRA = storage_values.get("C_AS_EXTRA", 0.0)   # may be 0 in some quarters
C_I_BASE  = storage_values.get("C_I_BASE")
C_E_BASE  = storage_values.get("C_E_BASE")

print(f"D_RS       = {D_RS}")       # expected: ~20-22 días
print(f"C_AS_BASE  = {C_AS_BASE}")  # expected: ~0.002525
print(f"C_AS_EXTRA = {C_AS_EXTRA}") # expected: ~0.000144752 or 0
print(f"C_I_BASE   = {C_I_BASE}")   # expected: ~0.096-0.141
print(f"C_E_BASE   = {C_E_BASE}")   # expected: ~0.071-0.076

D_RS       = 21.8
C_AS_BASE  = 0.002525
C_AS_EXTRA = 0.0
C_I_BASE   = 0.141516
C_E_BASE   = 0.101405


## 8 · Assemble Final constants Dict for Chosen RL Level

In [26]:
def get_constants_from_boe(
    peaje_level: str,
    boe_url: str,
    plant_constants: dict[str, float] | None = None,
) -> dict[str, float]:
    """Fetch BOE from URL and return constants dict for the selected RL level.

    Args:
        peaje_level: RL level such as "RL9".
        boe_url: Full BOE URL (txt/html page with tables).
        plant_constants: Optional installation-specific constants to merge.
            If None, PLANT_CONSTANTS is used.

    Returns:
        Dict with BOE-scraped constants + plant constants.
    """
    rl_level = peaje_level.upper().replace(" ", "").replace("-", "")
    if not rl_level.startswith("RL"):
        raise ValueError("peaje_level must look like 'RL9', 'RL6', etc.")

    base_constants = PLANT_CONSTANTS if plant_constants is None else plant_constants

    response = requests.get(boe_url, timeout=30)
    response.raise_for_status()
    soup_local = BeautifulSoup(response.text, "html.parser")
    tables = soup_local.find_all("table", class_="tabla")

    def _find_row_value(tbl_tag, keyword: str) -> float | None:
        for tr in tbl_tag.find_all("tr"):
            cells = [td.get_text(" ", strip=True) for td in tr.find_all(["th", "td"])]
            if len(cells) >= 2 and keyword.lower() in cells[0].lower():
                try:
                    return sp_float(cells[1])
                except (ValueError, IndexError):
                    pass
        return None

    tasa_cnmc = None
    tasa_gts = None
    tc_sa = None
    tv_sa = None

    for tbl in tables:
        v = _find_row_value(tbl, "CNMC")
        if v is not None:
            tasa_cnmc = v
        v = _find_row_value(tbl, "Gestor T")
        if v is not None:
            tasa_gts = v

        v = _find_row_value(tbl, "término fijo de capacidad del peaje de salida")
        if v is not None:
            tc_sa = v
        v = _find_row_value(tbl, "término variable de volumen del peaje de salida")
        if v is not None:
            tv_sa = v

    rl_levels = ["RL6", "RL7", "RL8", "RL9", "RL10", "RL11"]

    ocr_data: dict[str, float] = {}
    rl_data: dict[str, dict[str, float]] = {}

    for tbl in tables:
        df = table_to_df(tbl)
        if df.empty:
            continue

        table_text = " ".join(
            tbl.get_text(" ", strip=True).lower().split()[:120]
        )

        # OCR table only: "otros costes de regasificación"
        if "otros costes de regasificación" in table_text and df.shape[1] >= 2:
            for _, row in df.iterrows():
                label = str(row.iloc[0]).upper().replace(" ", "").replace("-", "")
                for rl in rl_levels:
                    if label.startswith(rl):
                        try:
                            ocr_data[rl] = sp_float(str(row.iloc[1]))
                        except ValueError:
                            pass

        # RL 4-column table: cargos unitarios + término fijo + término variable
        if df.shape[1] >= 4:
            for _, row in df.iterrows():
                label = str(row.iloc[0]).upper().replace(" ", "").replace("-", "")
                for rl in rl_levels:
                    if label.startswith(rl):
                        try:
                            rl_data[rl] = {
                                "TC_RL_J_BASE_2": sp_float(str(row.iloc[1])),
                                "TC_RL_J_BASE_1": sp_float(str(row.iloc[2])),
                                "TV_RL_J_BASE":   sp_float(str(row.iloc[3])),
                            }
                        except (ValueError, IndexError):
                            pass

    storage_keywords = {
        "días de existencias": "D_RS",
        "canon de almacenamiento": "C_AS_BASE",
        "prima de las subasta": "C_AS_EXTRA",
        "canon de inyección": "C_I_BASE",
        "canon de extracción": "C_E_BASE",
    }
    storage_values: dict[str, float] = {}
    for tbl in tables:
        for tr in tbl.find_all("tr"):
            cells = [td.get_text(" ", strip=True) for td in tr.find_all(["th", "td"])]
            if len(cells) < 2:
                continue
            desc = cells[0].lower()
            for kw, const_key in storage_keywords.items():
                if kw in desc and const_key not in storage_values:
                    try:
                        storage_values[const_key] = sp_float(cells[1])
                    except ValueError:
                        pass

    if rl_level not in rl_data:
        raise KeyError(f"RL level '{rl_level}' not found in BOE RL table. Available: {list(rl_data.keys())}")
    if rl_level not in ocr_data:
        raise KeyError(f"RL level '{rl_level}' not found in BOE OCR table. Available: {list(ocr_data.keys())}")

    boe_constants = {
        "TASA_CNMC": tasa_cnmc,
        "TASA_GTS": tasa_gts,
        "TC_SA": tc_sa,
        "TV_SA": tv_sa,
        "TC_OCR_J_BASE": ocr_data[rl_level],
        "TC_RL_J_BASE_2": rl_data[rl_level]["TC_RL_J_BASE_2"],
        "TC_RL_J_BASE_1": rl_data[rl_level]["TC_RL_J_BASE_1"],
        "TV_RL_J_BASE": rl_data[rl_level]["TV_RL_J_BASE"],
        "D_RS": storage_values.get("D_RS"),
        "C_AS_BASE": storage_values.get("C_AS_BASE"),
        "C_AS_EXTRA": storage_values.get("C_AS_EXTRA", 0.0),
        "C_I_BASE": storage_values.get("C_I_BASE"),
        "C_E_BASE": storage_values.get("C_E_BASE"),
    }

    missing = [k for k, v in boe_constants.items() if v is None]
    if missing:
        raise ValueError(f"Could not scrape required BOE constants: {missing}")

    return {**boe_constants, **base_constants}


# Build constants from BOE URL and selected RL level
constants = get_constants_from_boe(peaje_level=TARGET_RL_LEVEL, boe_url=BOE_URL)

print(f"\nconstants dict for RL level '{TARGET_RL_LEVEL}':")
for key, value in constants.items():
    print(f"  {key:<20} = {value}")


constants dict for RL level 'RL9':
  TASA_CNMC            = 0.14
  TASA_GTS             = 1.354
  TC_SA                = 0.091488
  TV_SA                = 0.0
  TC_OCR_J_BASE        = -0.011049
  TC_RL_J_BASE_2       = 0.00614
  TC_RL_J_BASE_1       = 0.160022
  TV_RL_J_BASE         = 0.000488
  D_RS                 = 21.8
  C_AS_BASE            = 0.002525
  C_AS_EXTRA           = 0.0
  C_I_BASE             = 0.141516
  C_E_BASE             = 0.101405
  Vc                   = 2.6059
  PCI                  = 0.9
  C_OYM_OTROS          = 20.491
  V_CO2                = 0.403
  RINV                 = 0.0
  V_IVPEE_RI           = 6.813
  V_AC                 = 0.0
  P_OTROS              = 0.0
  V_HV                 = 1.0289
  V_HF                 = 0.608095424114281
  I_HE                 = 0.0
  IVPEE                = 0.07


## 9 · Contrast Against `ro_calculation.config`

Diffs the scraped `constants` dict (cell above, requires network access to `BOE_URL`) against
the assembled constant set (`constants_for`), plus the reconciliation already done by hand
against the resolution PDF for q3_26 / IT-01144 (RL11):

- **`D_RS`**: code had `21.8` (stale); the resolution's "Número de días de existencias mínimas
  de seguridad" is `20.6` for q3_26. Fixed in `STATIC_CONSTANTS`.
- **`IVPEE`**: not a BOE-table constant at all — it's a quarterly tax rate reduced by successive
  Real Decreto-ley measures. Now driven by `data/IVPEE.xlsx`/`ivpee_for_quarter` in
  `ro_calculation/config.py` instead of being hardcoded (`4.9%` for q3_26).
- All RL11-dependent toll constants (`TC_OCR_J_BASE`, `TC_RL_J_BASE_1/2`, `TV_RL_J_BASE`) and the
  storage/transport constants (`TC_SA`, `TV_SA`, `C_AS_BASE/EXTRA`, `C_I_BASE`, `C_E_BASE`,
  `TASA_CNMC`, `TASA_GTS`) matched exactly — no other discrepancies found.

Verification: plugging the resolution's exact published inputs (`P_m=78.714`, `P_pvb=42.433`,
`P_co2=75.48`) into `calculate_ro` with the corrected constants reproduces the official
**IT-01144 RO = 55.377** to within a rounding cent (55.368).

In [ ]:
from ro_calculation import constants_for

code_constants = constants_for("q3_26")

rows = []
for key in sorted(set(constants) | set(code_constants)):
    boe_v = constants.get(key)
    code_v = code_constants.get(key)
    match = (
        "missing in code" if code_v is None else
        "missing in BOE scrape" if boe_v is None else
        "OK" if abs(boe_v - code_v) < 1e-6 else
        "MISMATCH"
    )
    rows.append({"constant": key, "scraped_from_boe": boe_v, "ro_calculation.config": code_v, "status": match})

df_diff = pd.DataFrame(rows)
print(df_diff.to_string(index=False))
mismatches = df_diff[df_diff["status"] != "OK"]
if mismatches.empty:
    print("\nNo discrepancies — ro_calculation.config matches the scraped BOE constants.")
else:
    print(f"\n{len(mismatches)} discrepancy(ies) found — see 'status' column above.")